# Phases C–H: Combined H100-Optimized Notebook

Paste these cells **after** the Phase B sweep notebook (after the diagnostic sweep completes and a winner is identified).

**H100 settings applied throughout:**
- `batch_size=192` · `effective_batch_size=768` · `grad_accum=2`
- `amp_dtype='bfloat16'` — native H100 bf16 tensor cores
- `num_workers=4` — real CPUs on H100 instance *(set to 0 for Colab H100)*
- `EVAL_BATCH_SIZE=384` — no-gradient eval fits far larger batches
- SDPA in `glada_model.py` auto-dispatches to FlashAttention v2 on H100
- TF32 + `cudnn.benchmark` set at startup

---


## 0. One-Time Setup (shared by all phases below)

Run once per runtime. All phases below assume this has already been executed.

In [ ]:
# ── Install dependencies ──
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

# # ── Mount Google Drive ──
# from google.colab import drive
# drive.mount('/content/drive')

# ── H100 startup flags ──
import torch
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True   # H100 TF32 matmul
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True           # auto-tune convolution kernels
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.0f} GB)')
    if 'H100' not in gpu_name:
        print('WARNING: Not an H100 — reduce BATCH_SIZE below if OOM')
else:
    print('WARNING: No GPU detected')


In [ ]:
# ── Copy GLADA package from Drive to local SSD ──
import shutil, os, sys

# DRIVE_PKG = "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code"
# LOCAL_PKG  = '/content/glada_code'

# if os.path.exists(LOCAL_PKG):
#     shutil.rmtree(LOCAL_PKG)
# shutil.copytree(DRIVE_PKG, LOCAL_PKG)
# print(f'Package copied to {LOCAL_PKG}')

# if LOCAL_PKG not in sys.path:
#     sys.path.insert(0, LOCAL_PKG)


In [ ]:
# # ── Dataset extraction (one archive at a time to conserve SSD) ──
# import os, shutil, tarfile, zipfile

# DRIVE_DATASETS = '/content/drive/MyDrive/Jakob/Datasets'
# LOCAL_DATA = '/content/data'

# def extract_one_dataset(drive_archive, extract_to, verify_path, name):
#     if os.path.isdir(verify_path) and os.listdir(verify_path):
#         print(f'  [{name}] Already extracted at {verify_path}')
#         return
#     archive_name = os.path.basename(drive_archive)
#     local_archive = f'/content/{archive_name}'
#     print(f'  [{name}] Copying {archive_name} to SSD...')
#     shutil.copy2(drive_archive, local_archive)
#     os.makedirs(extract_to, exist_ok=True)
#     print(f'  [{name}] Extracting...')
#     if archive_name.endswith('.zip'):
#         with zipfile.ZipFile(local_archive, 'r') as zf:
#             zf.extractall(extract_to)
#     elif archive_name.endswith(('.tar.gz', '.tgz')):
#         with tarfile.open(local_archive, 'r:gz') as tf:
#             tf.extractall(extract_to)
#     elif archive_name.endswith('.tar'):
#         with tarfile.open(local_archive, 'r:') as tf:
#             tf.extractall(extract_to)
#     os.remove(local_archive)
#     print(f'  [{name}] Done. Archive deleted from SSD.\n')

# for ds in [
#     dict(drive_archive=f'{DRIVE_DATASETS}/LA.tar',
#          extract_to=f'{LOCAL_DATA}/asvspoof',
#          verify_path=f'{LOCAL_DATA}/asvspoof/LA',
#          name='ASVspoof 2019 LA'),
#     dict(drive_archive=f'{DRIVE_DATASETS}/release_in_the_wild.tar',
#          extract_to=f'{LOCAL_DATA}/itw',
#          verify_path=f'{LOCAL_DATA}/itw/release_in_the_wild',
#          name='In-the-Wild'),
#     dict(drive_archive=f'{DRIVE_DATASETS}/musan.tar.gz',
#          extract_to=LOCAL_DATA,
#          verify_path=f'{LOCAL_DATA}/musan',
#          name='MUSAN'),
#     dict(drive_archive=f'{DRIVE_DATASETS}/rirs_noises.zip',
#          extract_to=LOCAL_DATA,
#          verify_path=f'{LOCAL_DATA}/RIRS_NOISES',
#          name='RIR'),
# ]:
#     extract_one_dataset(**ds)

# # ── Shared path constants (used by all phases) ──
# ASV_PROTO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
# ASV_AUDIO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
# ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
# ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
# ITW_META       = '/content/data/itw/release_in_the_wild/meta.csv'
# ITW_AUDIO      = '/content/data/itw/release_in_the_wild'
# MUSAN_PATH     = '/content/data/musan'
# RIR_PATH       = '/content/data/RIRS_NOISES/simulated_rirs'

# for p in [ASV_PROTO, ASV_AUDIO, ITW_META, MUSAN_PATH, RIR_PATH]:
#     assert os.path.exists(p), f'Missing: {p}'
# print('All dataset paths verified.')


In [ ]:
# ── Package imports (shared by all phases) ──
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
    create_dataloaders,
)
from glada.augmentation import CombinedAugmenter
from glada.trainers import GLADATrainer
from glada.models import GLADAModel
from glada.evaluation import (
    MultiDatasetEvaluator,
    average_checkpoints,
    save_averaged_checkpoint,
)

import copy, gc, glob, json, os, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch

matplotlib.rcParams.update({
    'font.size': 12, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

# ── H100 batch settings ──
# H100 SXM5: 80 GB HBM3.  Forward pass = batch_size source + batch_size target.
# grad_accum = effective_batch_size // (batch_size * 2) = 768 // 384 = 2
# Larger effective batch (768 vs 384) produces smoother domain adaptation gradients.
# Set num_workers=0 if running on Colab H100 (multiprocessing issues).
H100_BATCH_SIZE      = 192   # per-side; total forward = 384 samples
H100_EFFECTIVE_BATCH = 768   # grad_accum=2
H100_NUM_WORKERS     = 4     # real CPUs on H100 instance; set 0 for Colab H100
H100_EVAL_BATCH      = 384   # no gradients stored; H100 can handle 8x A100 eval batch
H100_AMP_DTYPE       = 'bfloat16'

print(f'GLADA package loaded. Backbones: {list(BACKBONE_REGISTRY.keys())}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'H100 config: batch={H100_BATCH_SIZE}, eff={H100_EFFECTIVE_BATCH}, '
      f'workers={H100_NUM_WORKERS}, eval_batch={H100_EVAL_BATCH}')


---
## Phase C: Full Training with Phase B Winner

Full-budget training (5 + 45 = 50 epochs) using the winning config identified by the
Phase B diagnostic sweep. Update `WINNER` below with values from `sweep_results.json`
(or from the `diag_results` dict still live in the Phase B session).


In [ ]:
# ── Load Phase B sweep results and pick winner ──
import json, os

SWEEP_RESULTS = '/content/drive/MyDrive/Jakob/Experiments/sweep_results.json'
DIAG_RESULTS  = '/content/drive/MyDrive/Jakob/Experiments/phase2_diagnostic_results.json'
FIXES_RESULTS = '/content/drive/MyDrive/Jakob/Experiments/phase2_fixes_results.json'

# Try to auto-load best config from Phase B sweep
winner = None
for path, key in [(SWEEP_RESULTS, None), (FIXES_RESULTS, None)]:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if isinstance(data, list) and data:
            # sweep_results is sorted by stability_score ascending (best first)
            winner = data[0]
            print(f'Loaded winner from {path}:')
            print(json.dumps(winner, indent=2, default=str))
            break

if winner is None:
    print('No sweep results found. Fill WINNER manually below.')

# ── Override manually if needed ──
# WINNER = {
#     'backbone_lr': 2e-6,
#     'freeze_bottom_n': 12,
#     'phase1_epochs': 5,
# }


In [ ]:
# ── Phase C config ──
SEED    = 42
BACKBONE = 'wavlm-large'

# Pull from winner dict or use safe defaults
backbone_lr  = winner.get('backbone_lr',      2e-6) if winner else 2e-6
freeze_n     = winner.get('freeze_bottom_n',  12)   if winner else 12
p1_epochs    = winner.get('phase1_epochs',    5)    if winner else 5

config_c = GLADAConfig(
    experiment_name=f'glada_C_{BACKBONE}_seed{SEED}',
    backbone=BACKBONE,
    seed=SEED,

    # ── Winner hyperparameters ──
    backbone_lr=backbone_lr,
    freeze_bottom_n_layers=freeze_n,
    phase1_epochs=p1_epochs,
    phase2_epochs=45,                 # Full Phase 2

    # ── H100 batch settings ──
    batch_size=H100_BATCH_SIZE,       # 192 per side (384 total forward)
    effective_batch_size=H100_EFFECTIVE_BATCH,  # grad_accum=2
    num_workers=H100_NUM_WORKERS,
    amp_dtype=H100_AMP_DTYPE,

    # ── Loss weights ──
    head_lr=1e-3,
    vib_beta=1e-2,
    domain_loss_weight=0.1,
    speaker_loss_weight=0.1,
    cka_loss_weight=0.01,

    # ── Augmentation ──
    rawboost_prob=0.5,
    musan_path=MUSAN_PATH,
    musan_prob=0.5,
    rir_path=RIR_PATH,
    rir_prob=0.5,

    # ── Full model ──
    use_glada=True,
    use_vib=True,
    use_speaker_adversary=True,
    use_cka_loss=True,
    use_hierarchical=True,
    use_rawboost=True,

    # ── Checkpointing ──
    save_every_n_epochs=1,
    save_top_k=10,
)

print(f'Phase C config:')
print(f'  backbone_lr={config_c.backbone_lr}, freeze_bottom={config_c.freeze_bottom_n_layers}')
print(f'  batch={config_c.batch_size}, eff={config_c.effective_batch_size}, '
      f'grad_accum={config_c.grad_accum_steps}')
print(f'  Total epochs: {config_c.total_epochs} '
      f'(Phase 1: {config_c.phase1_epochs}, Phase 2: {config_c.phase2_epochs})')


In [ ]:
# ── Phase C: Build datasets and train ──
set_seed(SEED)
augmenter_c = CombinedAugmenter(config_c)

source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO, audio_dir=ASV_AUDIO,
    config=config_c, augmenter=augmenter_c)
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=config_c, use_labels=False)
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=config_c, use_labels=True)
extra_eval = {}
if os.path.exists(ASV_EVAL_PROTO):
    extra_eval['2019_LA'] = ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO, audio_dir=ASV_EVAL_AUDIO, config=config_c)

loaders_c = create_dataloaders(config_c, source_ds, target_ds, eval_ds, extra_eval)

RESUME_FROM = None  # Set to checkpoint path to resume after disconnection

trainer_c = GLADATrainer(
    config=config_c,
    source_loader=loaders_c['source'],
    target_loader=loaders_c['target'],
    eval_loader=loaders_c['eval'],
    extra_eval_loaders={k: v for k, v in loaders_c.items()
                        if k not in ('source', 'target', 'eval')},
    num_speakers=source_ds.num_speakers,
    resume_from=RESUME_FROM,
)
results_c = trainer_c.train()
print(f'\nPhase C done. Best ITW EER: {results_c["best_eer"]:.2f}% (epoch {results_c["best_epoch"]})')


In [ ]:
# ── Phase C: Evaluation + checkpoint averaging ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

best_path = os.path.join(results_c['run_dir'], 'best_model.pt')
model_c = GLADAModel(config_c)
ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
model_c.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
model_c.to(device).eval()

eval_dict = {'ITW': loaders_c['eval']}
eval_dict.update({k: v for k, v in loaders_c.items()
                  if k not in ('source', 'target', 'eval')})
evaluator_c = MultiDatasetEvaluator(model_c, config_c, device, eval_dict)
all_results_c = evaluator_c.evaluate_all()
print(evaluator_c.results_to_markdown(all_results_c))

# Top-5 checkpoint averaging
try:
    avg_model = average_checkpoints(
        run_dir=results_c['run_dir'], model_class=GLADAModel,
        config=config_c, top_k=5, metric='eer',
        device='cuda' if torch.cuda.is_available() else 'cpu')
    evaluator_avg = MultiDatasetEvaluator(avg_model, config_c, device, eval_dict)
    avg_results = evaluator_avg.evaluate_all()
    avg_itw = avg_results['ITW']['eer']
    delta = results_c['best_eer'] - avg_itw
    print(f'\nCheckpoint avg (top-5): {avg_itw:.2f}% | Single best: {results_c["best_eer"]:.2f}% | Delta: {delta:+.2f}%')
    save_averaged_checkpoint(avg_model, results_c['run_dir'], config_c, 5, source_epochs=[])
except Exception as e:
    print(f'Checkpoint averaging failed: {e}')

# Save results
with open(os.path.join(results_c['run_dir'], 'eval_results.json'), 'w') as f:
    json.dump({k: {kk: float(vv) if hasattr(vv, 'item') else vv
                   for kk, vv in v.items()}
               for k, v in all_results_c.items()}, f, indent=2)

# Cleanup
del trainer_c, source_ds, target_ds, eval_ds, loaders_c
gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ── Phase C: Training curves ──
history = results_c['history']
epochs  = [h['epoch'] for h in history]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Phase C Training: {config_c.backbone} (seed={config_c.seed})', fontsize=14)

ax = axes[0,0]
ax.plot(epochs, [h['train']['total_loss'] for h in history], 'b-')
ax.axvline(x=config_c.phase1_epochs, color='r', linestyle='--', alpha=0.5, label='Phase 2')
ax.set(xlabel='Epoch', ylabel='Total Loss', title='Total Training Loss'); ax.legend()

ax = axes[0,1]
for key, label in [('spoof_loss','Task (CE)'),('glada_loss','GLADA (DA)'),('speaker_loss','Speaker')]:
    ax.plot(epochs, [h['train'][key] for h in history], label=label)
ax.axvline(x=config_c.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.set(xlabel='Epoch', ylabel='Loss', title='Loss Components'); ax.legend()

ax = axes[0,2]
ax.plot(epochs, [h['train']['kl_loss'] for h in history], label='KL (clamped)')
ax.plot(epochs, [h['train']['cka_loss'] for h in history], label='CKA')
ax.axvline(x=config_c.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.set(xlabel='Epoch', ylabel='Loss', title='KL + CKA Losses'); ax.legend()

ds_names = list(history[0]['eval'].keys()) if history else []
ax = axes[1,0]
for ds in ds_names:
    eers = [h['eval'][ds]['eer'] for h in history if ds in h['eval']]
    ax.plot(epochs[:len(eers)], eers, label=ds)
ax.axvline(x=config_c.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.set(xlabel='Epoch', ylabel='EER (%)', title='EER per Dataset'); ax.legend()

ax = axes[1,1]
for ds in ds_names:
    aucs = [h['eval'][ds]['auc'] for h in history if ds in h['eval']]
    ax.plot(epochs[:len(aucs)], aucs, label=ds)
ax.axvline(x=config_c.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.set(xlabel='Epoch', ylabel='AUC', title='AUC per Dataset'); ax.legend()

ax = axes[1,2]
ax.plot(epochs, [h['train']['spoof_acc']*100 for h in history], label='Task Acc')
ax.plot(epochs, [h['train']['speaker_acc']*100 for h in history], label='Speaker Acc')
ax.axvline(x=config_c.phase1_epochs, color='r', linestyle='--', alpha=0.5)
ax.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Task + Speaker Accuracy'); ax.legend()

plt.tight_layout()
fig_path = os.path.join(results_c['run_dir'], 'training_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {fig_path}')


---
## Phase D: Ablation Study

Leave-one-out ablations + baselines + augmentation + multi-backbone sweep.
Shares datasets already extracted above. All runs use the H100 batch settings.
`USE_FULL_EPOCHS=True` — H100 speed makes full-budget ablations feasible.


In [ ]:
# ── Ablation configuration matrix ──
USE_FULL_EPOCHS = True                               # H100 is fast enough for full epochs
ABLATION_PHASE1 = 5  if USE_FULL_EPOCHS else 3
ABLATION_PHASE2 = 45 if USE_FULL_EPOCHS else 17

BACKBONES = ['wavlm-large', 'xlsr-300m', 'data2vec-large', 'wav2vec2-robust', 'wavlm-base-plus']
SEEDS     = [42, 123, 2026]

MUSAN_AVAIL = MUSAN_PATH if os.path.isdir(MUSAN_PATH) else ''
RIR_AVAIL   = RIR_PATH   if os.path.isdir(RIR_PATH)   else ''

# ── Leave-one-out ablations ──
ABLATION_CONFIGS = {
    'full_glada':  {},
    'no_glada':    {'use_glada': False, 'use_global_domain_disc': True},
    'no_vib':      {'use_vib': False},
    'no_speaker':  {'use_speaker_adversary': False},
    'no_cka':      {'use_cka_loss': False},
    'no_rawboost': {'use_rawboost': False, 'rawboost_prob': 0.0},
    'no_hier':     {'use_hierarchical': False},
}

# ── Key baselines ──
BASELINE_CONFIGS = {
    'global_da_only':    {'use_glada': False, 'use_global_domain_disc': True, 'use_vib': True},
    'hierarchical_only': {'use_glada': False, 'use_global_domain_disc': False,
                          'use_vib': False, 'use_speaker_adversary': False},
    'no_da':             {'use_glada': False, 'use_global_domain_disc': False,
                          'use_speaker_adversary': False},
    'backbone_only':     {'use_glada': False, 'use_global_domain_disc': False,
                          'use_vib': False, 'use_speaker_adversary': False,
                          'use_cka_loss': False, 'use_rawboost': False, 'rawboost_prob': 0.0},
}

# ── Augmentation ablation ──
AUG_CONFIGS = {
    'rawboost_only':      {'musan_path': '', 'rir_path': ''},
    'rawboost_musan':     {'musan_path': MUSAN_AVAIL, 'rir_path': ''},
    'rawboost_musan_rir': {'musan_path': MUSAN_AVAIL, 'rir_path': RIR_AVAIL},
    'musan_rir_only':     {'use_rawboost': False, 'rawboost_prob': 0.0,
                           'musan_path': MUSAN_AVAIL, 'rir_path': RIR_AVAIL},
}

def make_ablation_config(name, backbone='wavlm-large', seed=42, overrides=None):
    overrides = overrides or {}
    cfg = GLADAConfig(
        experiment_name=f'ablation_{name}',
        backbone=backbone, seed=seed,
        phase1_epochs=ABLATION_PHASE1,
        phase2_epochs=ABLATION_PHASE2,
        # H100 settings
        batch_size=H100_BATCH_SIZE,
        effective_batch_size=H100_EFFECTIVE_BATCH,
        num_workers=H100_NUM_WORKERS,
        amp_dtype=H100_AMP_DTYPE,
        save_every_n_epochs=1,
        save_top_k=10,
    )
    for k, v in overrides.items():
        if hasattr(cfg, k):
            setattr(cfg, k, v)
    cfg.__post_init__()
    return cfg

# Build full experiment list
abl_experiments = []
# Leave-one-out on wavlm-large seed 42
for name, overrides in ABLATION_CONFIGS.items():
    abl_experiments.append(make_ablation_config(name, overrides=overrides))
# Baselines
for name, overrides in BASELINE_CONFIGS.items():
    abl_experiments.append(make_ablation_config(name, overrides=overrides))
# Augmentation ablation
for name, overrides in AUG_CONFIGS.items():
    abl_experiments.append(make_ablation_config(name, overrides=overrides))
# Multi-backbone (full GLADA only, seed 42)
for bb in BACKBONES[1:]:   # wavlm-large already covered above
    abl_experiments.append(make_ablation_config(f'full_{bb}', backbone=bb))
# Multi-seed (wavlm-large, top 2 configs, extra seeds)
for seed in SEEDS[1:]:
    abl_experiments.append(make_ablation_config('full_glada', seed=seed))
    abl_experiments.append(make_ablation_config('no_glada', seed=seed,
                                                 overrides={'use_glada': False,
                                                            'use_global_domain_disc': True}))

print(f'Ablation experiments: {len(abl_experiments)} total')
print(f'Epochs per run: {ABLATION_PHASE1}+{ABLATION_PHASE2}={ABLATION_PHASE1+ABLATION_PHASE2}')
print(f'H100 batch: {H100_BATCH_SIZE} per side, grad_accum={H100_EFFECTIVE_BATCH//(H100_BATCH_SIZE*2)}')


In [ ]:
# ── Load shared datasets for ablation (once) ──
_base_cfg_d = GLADAConfig(batch_size=H100_BATCH_SIZE, num_workers=H100_NUM_WORKERS)

source_ds_d = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO, audio_dir=ASV_AUDIO,
    config=_base_cfg_d, augmenter=None)
target_ds_d = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=_base_cfg_d, use_labels=False)
eval_ds_d = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=_base_cfg_d, use_labels=True)
extra_eval_d = {}
if os.path.exists(ASV_EVAL_PROTO):
    extra_eval_d['2019_LA'] = ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO, audio_dir=ASV_EVAL_AUDIO, config=_base_cfg_d)

NUM_SPEAKERS_D = source_ds_d.num_speakers
print(f'Source: {len(source_ds_d):,} samples, {NUM_SPEAKERS_D} speakers')
print(f'Target: {len(target_ds_d):,} samples | Eval: {len(eval_ds_d):,} samples')


In [ ]:
# ── Ablation sweep loop ──
abl_results = {}
sweep_start = time.time()

for idx, cfg in enumerate(abl_experiments, 1):
    print(f"\n{'='*60}")
    print(f"ABL {idx}/{len(abl_experiments)}: {cfg.experiment_name}")
    print(f"{'='*60}")

    # Resume support: skip if already completed
    existing = glob.glob(os.path.join(
        cfg.drive_log_dir, cfg.experiment_name, '*', 'history.json'))
    if existing:
        print(f"  SKIPPING — already completed: {existing[0]}")
        with open(existing[0]) as f:
            hist = json.load(f)
        abl_results[cfg.experiment_name] = {
            'best_eer': min(h.get('eval',{}).get('ITW',{}).get('eer',999) for h in hist),
            'status': 'loaded'}
        continue

    set_seed(cfg.seed)
    augmenter_d = CombinedAugmenter(cfg)
    source_ds_d.augmenter = augmenter_d   # swap augmenter in-place

    loaders_d = create_dataloaders(cfg, source_ds_d, target_ds_d, eval_ds_d, extra_eval_d)

    try:
        trainer_d = GLADATrainer(
            config=cfg,
            source_loader=loaders_d['source'],
            target_loader=loaders_d['target'],
            eval_loader=loaders_d['eval'],
            extra_eval_loaders={k: v for k, v in loaders_d.items()
                                if k not in ('source','target','eval')},
            num_speakers=NUM_SPEAKERS_D,
        )
        res = trainer_d.train()
        abl_results[cfg.experiment_name] = {
            'best_eer': res['best_eer'], 'best_epoch': res['best_epoch'],
            'run_dir': res['run_dir'], 'status': 'completed'}
        print(f"  Best ITW EER: {res['best_eer']:.2f}% (ep {res['best_epoch']})")
    except Exception as e:
        print(f"  FAILED: {e}")
        abl_results[cfg.experiment_name] = {'error': str(e), 'status': 'failed'}

    if 'trainer_d' in dir(): del trainer_d
    gc.collect(); torch.cuda.empty_cache()

    # Save incrementally
    with open(f'{cfg.drive_log_dir}/ablation_results.json', 'w') as f:
        json.dump(abl_results, f, indent=2, default=str)

elapsed = (time.time() - sweep_start) / 60
print(f"\nAblation sweep done in {elapsed:.1f} min")


In [ ]:
# ── Ablation results tables ──
import pandas as pd

# Leave-one-out table
rows = []
ref_eer = abl_results.get('ablation_full_glada', {}).get('best_eer')
for name, overrides in {**ABLATION_CONFIGS, **BASELINE_CONFIGS}.items():
    exp = f'ablation_{name}'
    r = abl_results.get(exp, {})
    eer = r.get('best_eer')
    delta = (eer - ref_eer) if (eer and ref_eer) else None
    rows.append({'Config': name,
                 'ITW EER (%)': f'{eer:.2f}' if eer else 'N/A',
                 'ΔEER': f'{delta:+.2f}' if delta is not None else 'N/A'})

df = pd.DataFrame(rows)
print('\n=== Leave-One-Out Ablation + Baselines ===')
print(df.to_string(index=False))

# Backbone comparison
print('\n=== Backbone Comparison ===')
bb_rows = [{'Backbone': 'wavlm-large',
             'ITW EER (%)': f'{abl_results.get("ablation_full_glada",{}).get("best_eer","N/A"):.2f}' if isinstance(abl_results.get("ablation_full_glada",{}).get("best_eer"), float) else 'N/A'}]
for bb in BACKBONES[1:]:
    exp = f'ablation_full_{bb}'
    r = abl_results.get(exp, {})
    eer = r.get('best_eer')
    bb_rows.append({'Backbone': bb, 'ITW EER (%)': f'{eer:.2f}' if eer else 'N/A'})
print(pd.DataFrame(bb_rows).to_string(index=False))


---
## Phase E: Multi-Seed Verification

Run full GLADA and no_glada at seeds 123 and 2026 (seed 42 already covered in Phase D).
Produces mean ± std for thesis reporting (Picard, 2021).


In [ ]:
# ── Multi-seed configs ──
# Seeds 123 and 2026 complement seed 42 from Phase D.
# Only the two most thesis-critical configs: full_glada and no_glada.
MULTISEED_RUNS = []
for seed in [123, 2026]:
    for name, overrides in [('full_glada', {}),
                             ('no_glada', {'use_glada': False, 'use_global_domain_disc': True})]:
        MULTISEED_RUNS.append(make_ablation_config(f'{name}_s{seed}',
                                                    seed=seed, overrides=overrides))

print(f'{len(MULTISEED_RUNS)} multi-seed runs:')
for c in MULTISEED_RUNS:
    print(f'  {c.experiment_name}: seed={c.seed}, glada={c.use_glada}')


In [ ]:
# ── Multi-seed training loop ──
# Reuses source_ds_d / target_ds_d / eval_ds_d from Phase D above.
multiseed_results = {}

for idx, cfg in enumerate(MULTISEED_RUNS, 1):
    print(f"\n{'='*60}")
    print(f"MULTISEED {idx}/{len(MULTISEED_RUNS)}: {cfg.experiment_name}")
    print(f"{'='*60}")

    existing = glob.glob(os.path.join(
        cfg.drive_log_dir, cfg.experiment_name, '*', 'history.json'))
    if existing:
        print(f"  SKIPPING — already completed: {existing[0]}")
        with open(existing[0]) as f:
            hist = json.load(f)
        multiseed_results[cfg.experiment_name] = {
            'best_eer': min(h.get('eval',{}).get('ITW',{}).get('eer',999) for h in hist),
            'status': 'loaded'}
        continue

    set_seed(cfg.seed)
    source_ds_d.augmenter = CombinedAugmenter(cfg)
    loaders_e = create_dataloaders(cfg, source_ds_d, target_ds_d, eval_ds_d, extra_eval_d)

    try:
        trainer_e = GLADATrainer(
            config=cfg,
            source_loader=loaders_e['source'],
            target_loader=loaders_e['target'],
            eval_loader=loaders_e['eval'],
            extra_eval_loaders={k: v for k, v in loaders_e.items()
                                if k not in ('source','target','eval')},
            num_speakers=NUM_SPEAKERS_D,
        )
        res = trainer_e.train()
        multiseed_results[cfg.experiment_name] = {
            'best_eer': res['best_eer'], 'best_epoch': res['best_epoch'], 'status': 'completed'}
        print(f"  Best ITW EER: {res['best_eer']:.2f}%")
    except Exception as e:
        print(f"  FAILED: {e}")
        multiseed_results[cfg.experiment_name] = {'error': str(e), 'status': 'failed'}

    if 'trainer_e' in dir(): del trainer_e
    gc.collect(); torch.cuda.empty_cache()

# Save
with open(f'/content/drive/MyDrive/Jakob/Experiments/multiseed_results.json', 'w') as f:
    json.dump(multiseed_results, f, indent=2, default=str)


In [ ]:
# ── Multi-seed analysis ──
all_seeds_results = {**abl_results, **multiseed_results}

groups = {
    'Full GLADA':  ['ablation_full_glada',   'ablation_full_glada_s123',  'ablation_full_glada_s2026'],
    'No GLADA':    ['ablation_no_glada',      'ablation_no_glada_s123',    'ablation_no_glada_s2026'],
    'No VIB':      ['ablation_no_vib'],
    'No Speaker':  ['ablation_no_speaker'],
    'No CKA':      ['ablation_no_cka'],
    'No Hier':     ['ablation_no_hier'],
    'No RawBoost': ['ablation_no_rawboost'],
    'Global DA':   ['ablation_global_da_only'],
}

print(f"{'Config':<20s} {'Seeds':>6s} {'Mean EER':>10s} {'Std':>8s} {'Values'}")
print('-' * 75)
plot_data = {}
for label, keys in groups.items():
    eers = [all_seeds_results[k]['best_eer']
            for k in keys if k in all_seeds_results
            and isinstance(all_seeds_results[k].get('best_eer'), float)]
    plot_data[label] = eers
    if not eers:
        print(f'  {label:<20s} {"0":>6s} {"N/A":>10s} {"N/A":>8s}')
    elif len(eers) == 1:
        print(f'  {label:<20s} {"1":>6s} {eers[0]:>9.2f}% {"—":>8s} {eers}')
    else:
        print(f'  {label:<20s} {len(eers):>6d} {np.mean(eers):>9.2f}% {np.std(eers):>7.2f}% {[f"{e:.2f}" for e in eers]}')

# Bar chart with error bars
plot_labels = [k for k, v in plot_data.items() if v]
plot_means  = [np.mean(plot_data[k]) for k in plot_labels]
plot_stds   = [np.std(plot_data[k])  if len(plot_data[k]) > 1 else 0 for k in plot_labels]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2196F3' if 'Full' in l else '#FF9800' for l in plot_labels]
ax.bar(range(len(plot_labels)), plot_means, yerr=plot_stds, capsize=6,
       color=colors, alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(plot_labels)))
ax.set_xticklabels(plot_labels, rotation=25, ha='right')
ax.set(ylabel='ITW EER (%)', title='Multi-Seed Results: Mean ± Std')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Jakob/Experiments/diagnostics/multiseed.png', dpi=150)
plt.show()


---
## Phase F: Multi-Dataset Evaluation

Evaluate best checkpoint across all 6 datasets. Extracts one dataset at a time
to stay within SSD budget. `EVAL_BATCH_SIZE=384` (H100 has no gradient storage
overhead during eval — fits far larger batches than during training).


In [ ]:
# ── Load best model from Phase C ──
from glada.datasets.asvspoof2021 import ASVspoof2021Dataset
from glada.datasets.wavefake import WaveFakeDataset
from torch.utils.data import DataLoader
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_BATCH_SIZE = H100_EVAL_BATCH   # 384 — no gradient memory on H100 during eval

# ── Point to best run from Phase C ──
# If running after Phase C in the same session, use results_c.
# Otherwise, set BEST_RUN_DIR manually:
# BEST_RUN_DIR = '/content/drive/MyDrive/Jakob/Experiments/glada_C_wavlm-large_seed42/20260414_...'

try:
    BEST_RUN_DIR = results_c['run_dir']
    config_f = config_c
    print(f'Using Phase C run: {BEST_RUN_DIR}')
except NameError:
    # Load from Drive if Phase C ran in a previous session
    BEST_RUN_DIR = sorted(glob.glob(
        '/content/drive/MyDrive/Jakob/Experiments/glada_C_*/*/'))[-1]
    with open(os.path.join(BEST_RUN_DIR, 'config.json')) as f:
        cfg_dict = json.load(f)
    config_f = GLADAConfig(**{k: v for k, v in cfg_dict.items()
                               if hasattr(GLADAConfig, k)})
    config_f.__post_init__()
    print(f'Loaded config from {BEST_RUN_DIR}')

best_path = os.path.join(BEST_RUN_DIR, 'best_model.pt')
model_f = GLADAModel(config_f)
ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
model_f.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
model_f.to(device).eval()
print(f'Model loaded from {best_path}')


In [ ]:
# ── Sequential extract → eval → cleanup for each dataset ──
# One dataset on SSD at a time to stay within the disk budget.

DRIVE_BASE    = '/content/drive/MyDrive/Jakob/Datasets'
CODECFAKE_BASE = '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus'
LOCAL_ROOT    = '/content/data'
eval_summary  = {}

def run_eval_on_loader(loader, name):
    ev = MultiDatasetEvaluator(model_f, config_f, device, {name: loader})
    res = ev.evaluate_all()
    return res[name]

# ── ITW (already extracted from setup cell) ──
print('\n--- Evaluating ITW ---')
itw_loader = DataLoader(
    InTheWildDataset(ITW_META, ITW_AUDIO, config=config_f, use_labels=True),
    batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=H100_NUM_WORKERS, pin_memory=True)
eval_summary['ITW'] = run_eval_on_loader(itw_loader, 'ITW')
del itw_loader; gc.collect()
print(f"  ITW EER: {eval_summary['ITW']['eer']:.2f}%")

# ── ASVspoof 2019 LA eval (already extracted) ──
print('\n--- Evaluating ASVspoof 2019 LA eval ---')
asv19_loader = DataLoader(
    ASVspoofEvalDataset(ASV_EVAL_PROTO, ASV_EVAL_AUDIO, config=config_f),
    batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=H100_NUM_WORKERS, pin_memory=True)
eval_summary['2019_LA'] = run_eval_on_loader(asv19_loader, '2019_LA')
del asv19_loader; gc.collect()
print(f"  2019_LA EER: {eval_summary['2019_LA']['eer']:.2f}%")

# ── WaveFake ──
print('\n--- Evaluating WaveFake ---')
wf_archive = os.path.join(DRIVE_BASE, 'WaveFake.tar.gz')
wf_dir     = os.path.join(LOCAL_ROOT, 'wavefake')
if os.path.exists(wf_archive) and not os.path.exists(wf_dir):
    import tarfile
    with tarfile.open(wf_archive, 'r:gz') as tf:
        tf.extractall(LOCAL_ROOT)
if os.path.exists(wf_dir):
    wf_loader = DataLoader(
        WaveFakeDataset(wf_dir, max_len=config_f.max_audio_len),
        batch_size=EVAL_BATCH_SIZE, shuffle=False,
        num_workers=H100_NUM_WORKERS, pin_memory=True)
    eval_summary['WaveFake'] = run_eval_on_loader(wf_loader, 'WaveFake')
    del wf_loader; gc.collect()
    import shutil; shutil.rmtree(wf_dir)    # free SSD
    print(f"  WaveFake EER: {eval_summary['WaveFake']['eer']:.2f}%")
else:
    print('  WaveFake archive not found — skipping')

# Save intermediate results
with open(os.path.join(BEST_RUN_DIR, 'multidataset_eval.json'), 'w') as f:
    json.dump({k: {kk: float(vv) if hasattr(vv, 'item') else vv
                   for kk, vv in v.items()}
               for k, v in eval_summary.items()}, f, indent=2)

# ── Summary ──
print('\n=== Multi-Dataset Evaluation Summary ===')
print(f"{'Dataset':<20s} {'EER (%)':>10s} {'AUC':>10s}")
print('-' * 42)
for ds, r in eval_summary.items():
    print(f"  {ds:<18s} {r.get('eer','N/A'):>9.2f}% {r.get('auc','N/A'):>9.4f}")


---
## Phase G: Training Diagnostics

Post-hoc analysis of training dynamics. Reads `history.json` from Drive —
no GPU needed (pure analysis). Run after Phase C completes.


In [ ]:
# ── Load experiment history ──
EXP_DIR = '/content/drive/MyDrive/Jakob/Experiments'

def load_history(experiment_name):
    for pattern in [
        os.path.join(EXP_DIR, experiment_name, '*', 'history.json'),
        os.path.join(EXP_DIR, experiment_name, 'history.json'),
    ]:
        matches = glob.glob(pattern)
        if matches:
            with open(sorted(matches)[-1]) as f:
                return json.load(f)
    return None

# Auto-discover full GLADA run
history_g = None
for name in ['glada_C_wavlm-large_seed42', 'ablation_full_glada',
             'glada_wavlm-large_seed42', 'glada_final_run']:
    history_g = load_history(name)
    if history_g:
        print(f'Loaded {len(history_g)} epochs from: {name}')
        break

if history_g is None:
    print('No history found. Check EXP_DIR or set history_g manually.')

DIAG_DIR = os.path.join(EXP_DIR, 'diagnostics')
os.makedirs(DIAG_DIR, exist_ok=True)


In [ ]:
# ── Investigation 1: Loss weight balance ──
if history_g:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Investigation 1: Loss Weight Balance', fontsize=14)

    epochs_g = [h['epoch'] for h in history_g]
    spoof  = [h['train']['spoof_loss']  for h in history_g]
    glada  = [h['train']['glada_loss']  for h in history_g]
    spk    = [h['train']['speaker_loss'] for h in history_g]
    kl     = [h['train']['kl_loss']     for h in history_g]
    cka    = [h['train']['cka_loss']    for h in history_g]
    total  = [h['train']['total_loss']  for h in history_g]

    ax = axes[0]
    ax.stackplot(epochs_g, spoof, glada, spk, kl, cka,
                 labels=['Task','GLADA','Speaker','KL','CKA'], alpha=0.7)
    ax.set(xlabel='Epoch', ylabel='Loss', title='Stacked Loss Components')
    ax.legend(loc='upper right', fontsize=9)
    ax.axvline(x=5, color='r', linestyle='--', alpha=0.5, label='Phase 2')

    ax = axes[1]
    for vals, label in zip([spoof,glada,spk,kl,cka],
                            ['Task','GLADA','Speaker','KL','CKA']):
        pct = [v/t*100 if t else 0 for v,t in zip(vals,total)]
        ax.plot(epochs_g, pct, label=label)
    ax.axvline(x=5, color='r', linestyle='--', alpha=0.5)
    ax.set(xlabel='Epoch', ylabel='% of Total Loss', title='Relative Loss Contributions')
    ax.legend(fontsize=9)

    ax = axes[2]
    kl_raw = [h['train'].get('kl_raw', h['train']['kl_loss']) for h in history_g]
    ax.plot(epochs_g, kl_raw, label='KL raw')
    ax.plot(epochs_g, kl, label='KL clamped')
    ax.axvline(x=5, color='r', linestyle='--', alpha=0.5)
    ax.set(xlabel='Epoch', ylabel='KL Loss', title='KL: Raw vs Clamped (VIB health)')
    ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(DIAG_DIR, 'investigation1_loss_balance.png'), dpi=150)
    plt.show()


In [ ]:
# ── Investigation 2: GRL alpha schedule + EER vs adversarial strength ──
if history_g:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Investigation 2: Adversarial Dynamics', fontsize=14)

    domain_alphas  = [h.get('domain_alpha', h.get('grl_alpha', None)) for h in history_g]
    speaker_alphas = [h.get('speaker_alpha', None) for h in history_g]
    itw_eers = [h['eval']['ITW']['eer'] for h in history_g if 'ITW' in h.get('eval',{})]

    ax = axes[0]
    if any(a is not None for a in domain_alphas):
        ax.plot(epochs_g, domain_alphas, label='Domain α', color='blue')
    if any(a is not None for a in speaker_alphas):
        ax.plot(epochs_g, speaker_alphas, label='Speaker α', color='orange')
    ax.axvline(x=5, color='r', linestyle='--', alpha=0.5, label='Phase 2')
    ax.set(xlabel='Epoch', ylabel='GRL Alpha', title='GRL Alpha Schedule')
    ax.legend()

    ax = axes[1]
    ax.plot(epochs_g[:len(itw_eers)], itw_eers, 'g-o', ms=4, label='ITW EER')
    ax.axvline(x=5, color='r', linestyle='--', alpha=0.5, label='Phase 2')
    ax.set(xlabel='Epoch', ylabel='ITW EER (%)', title='ITW EER over Training')
    ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(DIAG_DIR, 'investigation2_adversarial.png'), dpi=150)
    plt.show()


In [ ]:
# ── Investigation 5: Training stability ──
if history_g:
    p1 = [h for h in history_g if h['phase'] == 1]
    p2 = [h for h in history_g if h['phase'] == 2]

    def eer_trend(phase_hist):
        eers = [h['eval']['ITW']['eer'] for h in phase_hist if 'ITW' in h.get('eval',{})]
        if len(eers) < 2:
            return 0.0
        return eers[-1] - eers[0]

    print('=== Training Stability ===')
    print(f'Phase 1 EER trend: {eer_trend(p1):+.2f}% (positive = degrading)')
    print(f'Phase 2 EER trend: {eer_trend(p2):+.2f}%')
    print(f'Phase 2 stable: {eer_trend(p2) <= 0}')

    if p2:
        p2_eers = [h['eval']['ITW']['eer'] for h in p2 if 'ITW' in h.get('eval',{})]
        print(f'Phase 2 EERs: {[f"{e:.2f}" for e in p2_eers]}')
        print(f'Phase 2 best: {min(p2_eers):.2f}% | last: {p2_eers[-1]:.2f}%')


---
## Phase H: Analysis & Thesis Figures

Generate all figures and tables for thesis Chapter 5. Pure analysis — no GPU needed.
Prerequisites: Phases C, D, E, F must have completed and saved results to Drive.


In [ ]:
# ── Auto-discover all experiment results ──
EXP_DIR = '/content/drive/MyDrive/Jakob/Experiments'
FIGURES_DIR = os.path.join(EXP_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

def load_best_eer(experiment_name):
    for pattern in [
        os.path.join(EXP_DIR, experiment_name, '*', 'history.json'),
        os.path.join(EXP_DIR, experiment_name, 'history.json'),
    ]:
        matches = glob.glob(pattern)
        if matches:
            with open(sorted(matches)[-1]) as f:
                hist = json.load(f)
            eers = [h.get('eval',{}).get('ITW',{}).get('eer',999) for h in hist]
            return min(eers) if eers else None
    return None

# Load ablation results
abl_path = os.path.join(EXP_DIR, 'ablation_results.json')
if os.path.exists(abl_path):
    with open(abl_path) as f:
        abl_data = json.load(f)
    print(f'Loaded ablation results: {len(abl_data)} experiments')
else:
    abl_data = {}
    print('No ablation_results.json found. Run Phase D first.')

# Load multi-dataset eval
eval_path = glob.glob(os.path.join(EXP_DIR, 'glada_C_*/*/multidataset_eval.json'))
multidataset = {}
if eval_path:
    with open(sorted(eval_path)[-1]) as f:
        multidataset = json.load(f)
    print(f'Loaded multi-dataset eval: {list(multidataset.keys())}')


In [ ]:
# ── Figure 1: Ablation bar chart ──
abl_order = ['full_glada', 'no_glada', 'global_da_only', 'no_vib', 'no_speaker',
             'no_cka', 'no_hier', 'no_rawboost', 'backbone_only']
abl_labels = ['Full GLADA', '–GLADA', 'Global DA', '–VIB', '–Speaker',
              '–CKA', '–Hier', '–RawBoost', 'Backbone only']

abl_eers = []
for name in abl_order:
    r = abl_data.get(f'ablation_{name}', {})
    eer = r.get('best_eer')
    abl_eers.append(eer if isinstance(eer, float) else None)

mask = [e is not None for e in abl_eers]
labels_f = [l for l, m in zip(abl_labels, mask) if m]
eers_f   = [e for e, m in zip(abl_eers,  mask) if m]
ref_eer  = abl_data.get('ablation_full_glada', {}).get('best_eer', None)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2196F3' if l == 'Full GLADA' else
          '#F44336' if l == '–GLADA'    else '#90A4AE'
          for l in labels_f]
bars = ax.barh(range(len(labels_f)), eers_f, color=colors, edgecolor='black', linewidth=0.5)
for i, (bar, eer) in enumerate(zip(bars, eers_f)):
    ax.text(eer + 0.1, bar.get_y() + bar.get_height()/2,
            f'{eer:.2f}%', va='center', fontsize=10)
if ref_eer:
    ax.axvline(x=ref_eer, color='blue', linestyle='--', alpha=0.7, label=f'Full GLADA ({ref_eer:.2f}%)')
    ax.legend()
ax.set_yticks(range(len(labels_f)))
ax.set_yticklabels(labels_f)
ax.set(xlabel='ITW EER (%)', title='Ablation Study: Component Contributions (lower = better)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_ablation.png'), dpi=300)
plt.show()
print(f"Saved: {os.path.join(FIGURES_DIR, 'fig_ablation.png')}")


In [ ]:
# ── Figure 2: Multi-dataset evaluation ──
if multidataset:
    ds_names_h = list(multidataset.keys())
    ds_eers_h  = [multidataset[d].get('eer', 0) for d in ds_names_h]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(ds_names_h, ds_eers_h, color='#2196F3', edgecolor='black', linewidth=0.5)
    for bar, eer in zip(bars, ds_eers_h):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{eer:.2f}%', ha='center', va='bottom', fontsize=10)
    ax.set(ylabel='EER (%)', title='GLADA: Multi-Dataset Generalization (lower = better)')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'fig_multidataset.png'), dpi=300)
    plt.show()


In [ ]:
# ── Figure 3: SOTA comparison table ──
# Published EER values on ITW (Muller et al., Interspeech 2022)
sota = [
    ('Wav2Vec2-Large + LCNN',   'Kang et al., 2024',     9.34),
    ('RawGAT-ST',               'Tak et al., 2021',       7.67),
    ('HierCon (WavLM-BP)',      'Liang et al., 2026',     6.87),
    ('RawBMamba',               'Xie et al., 2024',       5.83),
    ('IB-CAAN (WavLM-Large)',   'Huang et al., 2025',     4.61),
    ('GLADA (ours, WavLM-L)',   'This work',
     multidataset.get('ITW', {}).get('eer', None) or abl_data.get('ablation_full_glada',{}).get('best_eer',999)),
]

print('=== SOTA Comparison (ITW EER %) ===')
print(f"{'Model':<35s} {'Reference':<25s} {'ITW EER':>10s}")
print('-' * 75)
for model, ref, eer in sota:
    marker = ' <<' if model.startswith('GLADA') else ''
    print(f"  {model:<33s} {ref:<25s} {eer:>9.2f}%{marker}")

# Bar chart
fig, ax = plt.subplots(figsize=(11, 4))
models = [s[0] for s in sota]
eers   = [s[2] for s in sota]
colors = ['#F44336' if 'ours' in m else '#90A4AE' for m in models]
bars   = ax.bar(range(len(models)), eers, color=colors, edgecolor='black', linewidth=0.5)
for bar, eer in zip(bars, eers):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{eer:.2f}%', ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
ax.set(ylabel='ITW EER (%)', title='State-of-the-Art Comparison (lower = better)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_sota.png'), dpi=300)
plt.show()


In [ ]:
# ── Export summary JSON for LaTeX ──
summary = {
    'full_glada_itw_eer':  abl_data.get('ablation_full_glada',{}).get('best_eer'),
    'no_glada_itw_eer':    abl_data.get('ablation_no_glada',{}).get('best_eer'),
    'multidataset':        multidataset,
    'sota':                [{'model': s[0], 'ref': s[1], 'eer': s[2]} for s in sota],
}

summary_path = os.path.join(EXP_DIR, 'thesis_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Summary saved to {summary_path}')

# Zip figures for download
import shutil
zip_path = shutil.make_archive('/content/thesis_figures', 'zip', FIGURES_DIR)
print(f'Figures zipped: {zip_path}')
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print(f'Download: {zip_path}')
